# 02 · NDVI y condición de la vegetación

**Objetivo:** Mapear vigor fotosintético y resumir el NDVI medio.

**Datos:** Sentinel-2 SR: bandas B8 y B4.

**Relevancia para política ambiental y social:** Apoya monitoreo forestal, agrícola y de restauración.

**Limitaciones:** NDVI se satura en bosques densos y no mide biomasa directamente.


In [ ]:
# Instalar dependencias en Google Colab
!pip -q install earthengine-api geemap

import ee
import geemap
import datetime

ee.Authenticate()
ee.Initialize(project="TU_PROYECTO_GEE")

# Área de estudio de ejemplo: entorno de Chachapoyas, Amazonas, Perú
# Reemplázala por un polígono, activo de Earth Engine o coordenadas propias.
aoi = ee.Geometry.Point([-77.87, -6.23]).buffer(30000)

Map = geemap.Map()
Map.centerObject(aoi, 9)


In [ ]:
def mask_s2_sr(image):
    scl = image.select("SCL")
    clear = (
        scl.neq(3)   # sombra
        .And(scl.neq(8))  # nube media
        .And(scl.neq(9))  # nube alta
        .And(scl.neq(10)) # cirrus
        .And(scl.neq(11)) # nieve/hielo
    )
    return image.updateMask(clear).divide(10000).copyProperties(
        image, ["system:time_start"]
    )

def s2_composite(start, end):
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
        .map(mask_s2_sr)
        .median()
        .clip(aoi)
    )


In [ ]:
image = s2_composite("2026-01-01", "2026-07-29")
ndvi = image.normalizedDifference(["B8","B4"]).rename("NDVI")

Map.addLayer(ndvi, {"min":-0.2,"max":0.9,"palette":["brown","yellow","green"]}, "NDVI")
stats = ndvi.reduceRegion(ee.Reducer.mean(), aoi, 10, maxPixels=1e9)
print("NDVI medio:", stats.get("NDVI").getInfo())
Map
